# K083, K084, K085
# Self-Healing Cyber Network with IDS and GenAI Explainer

**Self-Healing Hybrid Intrusion Detection System (HIDS)** that
1. detects attacks with a neural-network IDS,
2. hardens itself against adversarial (FGSM) evasion,
3. explains every alert with a GenAI explainer,
4. retrains itself on what it missed (self-healing loop),
5. and **heals the damaged network** (isolate, fail over, restore, verify) with a written incident report.

Run the cells top to bottom (Runtime → Run all). Default dataset is NSL-KDD (downloads automatically). Switch to CIC-IDS2017 in the config cell to link the IDS alerts to the network map.

## Pipeline at a glance

```
Load data → Preprocess → Baseline IDS → FGSM attack → Adversarial training → Robust IDS
        → Live traffic monitor → GenAI explain → Log incident → Retrain → Redeploy  (repeat = self-healing)
        → Map alerts to hosts → Contain → Fail over → Restore → Verify → GenAI incident report
```

## 0. Setup and configuration

In [ ]:
import os, time, json, copy, random, glob, urllib.request, urllib.error
from dataclasses import dataclass, field
from typing import List, Optional, Dict
from collections import deque as _deque

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
try:
    from IPython.display import display
except Exception:
    display = print

# ---------------- settings you can change ----------------
DATASET        = "nsl"     # "nsl" = NSL-KDD (auto-download) | "cic" = CIC-IDS2017 TrafficLabelling CSVs (has IP addresses)
TRAIN_SAMPLE   = 20000     # rows used for training
EPOCHS         = 25        # training epochs
EPSILON        = 0.25      # FGSM strength (in standard deviations; features are standardised)
CONF_THRESHOLD = 0.85      # below this confidence a sample is "unknown attack"
CYCLES         = 4         # self-healing loop iterations
BATCH_SIZE     = 1500      # live samples per cycle
SEED           = 42

# CIC-IDS2017 only: where the day files are (Google Drive folder) or set CIC_MODE = "upload"
CIC_MODE   = "drive"
CIC_FOLDER = "/content/drive/MyDrive/CIC-IDS2017"

# Optional: Gemini key for AI-written summaries. Leave empty to use the built-in local explainer.
os.environ["GEMINI_API_KEY"] = ""
LLM_MODEL_GEMINI = "gemini-2.0-flash"
# ----------------------------------------------------------
PRIMARY, MUTED = "#12263f", "#6b7785"
random.seed(SEED); np.random.seed(SEED)
DATA_DIR = "data"; os.makedirs(DATA_DIR, exist_ok=True)

BASE_URL = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
NSL_KDD_COLUMNS = [
    "duration", "protocol_type", "service", "flag", "src_bytes", "dst_bytes",
    "land", "wrong_fragment", "urgent", "hot", "num_failed_logins", "logged_in",
    "num_compromised", "root_shell", "su_attempted", "num_root", "num_file_creations",
    "num_shells", "num_access_files", "num_outbound_cmds", "is_host_login",
    "is_guest_login", "count", "srv_count", "serror_rate", "srv_serror_rate",
    "rerror_rate", "srv_rerror_rate", "same_srv_rate", "diff_srv_rate",
    "srv_diff_host_rate", "dst_host_count", "dst_host_srv_count",
    "dst_host_same_srv_rate", "dst_host_diff_srv_rate", "dst_host_same_src_port_rate",
    "dst_host_srv_diff_host_rate", "dst_host_serror_rate", "dst_host_srv_serror_rate",
    "dst_host_rerror_rate", "dst_host_srv_rerror_rate", "label", "difficulty",
]
CATEGORICAL = ["protocol_type", "service", "flag"]
NSL_ORIG_COLUMNS, NSL_ORIG_CAT = list(NSL_KDD_COLUMNS), list(CATEGORICAL)

## 1. Data loading
Two sources: **NSL-KDD** (classic benchmark; its test file contains attack types never seen in training, so it acts as "live" zero-day traffic) and **CIC-IDS2017** (modern traffic with source/destination IPs, which lets alerts be placed on the network map).
`build_frame` turns any table into model-ready columns and **removes label-leaking columns** (for example `attack_cat`, `id`) so accuracy is honest.

In [ ]:
NORMAL_WORDS = {"normal", "benign", "0", "legitimate", "clean", "none", "no", "false", "background"}
LABEL_HINTS = ("label", "class", "attack", "target", "category", "result")
# Columns that leak the answer or are just row identifiers (UNSW-NB15: id, attack_cat, label ...)
LEAK_NAMES = {"id", "rowid", "row_id", "index", "unnamed: 0", "attack_cat", "attack_category", "attack_type",
              "attack", "label", "class", "category", "target", "difficulty"}
ID_NAMES = {"id", "rowid", "row_id", "index", "unnamed: 0", "srcip", "dstip", "sport", "dsport", "stime", "ltime",
            # CIC-IDS2017 identifiers (kept out of the model, used only to place alerts on the network map)
            "flow id", "source ip", "destination ip", "src ip", "dst ip", "src_ip", "dst_ip", "timestamp",
            "source port", "src port", "srcport", "sourceip", "destinationip", "id.orig_h", "id.resp_h"}
SRC_NAMES = {"source ip", "src ip", "srcip", "src_ip", "sourceip", "id.orig_h"}
DST_NAMES = {"destination ip", "dst ip", "dstip", "dst_ip", "destinationip", "id.resp_h"}


def _find_col(df, names):
    for c in df.columns:
        if str(c).lower().strip() in names:
            return c
    return None


def _read_csv(path, **kw):
    """CIC-IDS2017 files are not UTF-8 (the 'Web Attack – ...' dash), so fall back to latin-1."""
    try:
        return pd.read_csv(path, low_memory=False, **kw)
    except UnicodeDecodeError:
        return pd.read_csv(path, low_memory=False, encoding="latin-1", **kw)


def read_table(path, nsl_columns):
    """Reads a CSV/TXT. Auto-detects the headerless NSL-KDD format. Strips header whitespace (CIC-IDS2017)."""
    probe = _read_csv(path, header=None, nrows=2)
    if probe.shape[1] == len(nsl_columns) and str(probe.iloc[0, 1]).lower() in ("tcp", "udp", "icmp"):
        df = _read_csv(path, names=nsl_columns)
        return df.drop(columns=["difficulty"], errors="ignore"), True
    df = _read_csv(path)
    df.columns = [str(c).strip() for c in df.columns]
    return df, False


def guess_label_col(cols):
    low = {c.lower().strip(): c for c in cols}
    for h in LABEL_HINTS:
        for k, c in low.items():
            if k == h or k.endswith(h):
                return c
    return cols[-1]


def guess_normals(series):
    vals = [str(v).strip() for v in pd.Series(series).dropna().unique()]
    hit = [v for v in vals if v.lower() in NORMAL_WORDS]
    return ", ".join(hit) if hit else ""


def build_frame(df, label_col, normals, feat_cols=None, cats=None):
    """Turns any table into the layout the base classes expect (features + label + binary_label)."""
    df = df.copy()
    s = df[label_col]
    if pd.api.types.is_numeric_dtype(s) and (s.dropna() % 1 == 0).all():
        lab = s.fillna(-1).astype(int).astype(str)
    else:
        lab = s.astype(str).str.strip()
    if feat_cols is None:
        feat_cols, cats = [], []
        dropped = []
        y_str = s.astype(str).str.strip()
        for c in df.columns:
            if c == label_col:
                continue
            cl = str(c).lower().strip()
            # 1) known label-companions / row ids (e.g. UNSW-NB15 'attack_cat', 'label', 'id')
            if cl in LEAK_NAMES or cl in ID_NAMES:
                dropped.append(c)
                continue
            col = df[c]
            # 2) automatic leak guard: a non-numeric column that determines the label (near) perfectly
            if not pd.api.types.is_numeric_dtype(col) and col.nunique() > 1:
                purity = df.groupby(col.astype(str))[label_col].agg(lambda x: x.astype(str).value_counts(normalize=True).iloc[0])
                w = col.astype(str).value_counts(normalize=True)
                if float((purity * w.reindex(purity.index)).sum()) > 0.995 and y_str.nunique() > 1 and col.nunique() >= y_str.nunique():
                    dropped.append(c)
                    continue
            if pd.api.types.is_numeric_dtype(col) or pd.api.types.is_bool_dtype(col):
                if col.notna().any():
                    feat_cols.append(c)
            else:
                conv = pd.to_numeric(col, errors="coerce")
                if conv.notna().mean() > 0.95:
                    df[c] = conv
                    feat_cols.append(c)
                elif col.nunique() <= 100:
                    cats.append(c)
                    feat_cols.append(c)
        if dropped:
            print(f"[Dataset] Excluded leakage/ID columns: {dropped}")
        if not feat_cols:
            raise ValueError("No usable feature columns found in this file.")
    missing = [c for c in feat_cols if c not in df.columns]
    if missing:
        raise ValueError(f"File is missing training columns: {missing[:8]}")
    for c in feat_cols:
        if c in cats:
            df[c] = df[c].fillna("NA").astype(str)
        else:
            v = pd.to_numeric(df[c], errors="coerce").replace([np.inf, -np.inf], np.nan)
            df[c] = v.fillna(v.median() if v.notna().any() else 0).astype(float)
    out = df[feat_cols].copy()
    sc, dc = _find_col(df, SRC_NAMES), _find_col(df, DST_NAMES)
    if sc is not None and dc is not None:          # host info rides along (never a model feature)
        out["_src"], out["_dst"] = df[sc].astype(str).str.strip().values, df[dc].astype(str).str.strip().values
    out["label"] = lab.values
    out["binary_label"] = (~lab.str.lower().isin(normals)).astype(int).values
    return out, feat_cols, cats

In [ ]:
def load_nsl(cols, n, seed):
    """Downloads NSL-KDD (cached in ./data). Train file -> training sample, test file -> unseen live stream."""
    import urllib.request
    base = "https://raw.githubusercontent.com/defcom17/NSL_KDD/master/"
    frames = []
    for url, name in ((base + "KDDTrain%2B.txt", "KDDTrain+.txt"), (base + "KDDTest%2B.txt", "KDDTest+.txt")):
        path = os.path.join(DATA_DIR, name)
        if not os.path.exists(path):
            print(f"[Data] Downloading {name} ...")
            urllib.request.urlretrieve(url, path)
        df = pd.read_csv(path, names=cols)
        df["binary_label"] = (df["label"] != "normal").astype(int)
        frames.append(df)
    train, live = frames
    t = train.sample(n=min(n, len(train)), random_state=seed).reset_index(drop=True)
    l = live.sample(frac=1.0, random_state=seed).reset_index(drop=True)
    print(f"[Data] NSL-KDD: {len(t)} training rows ({int(t['binary_label'].sum())} attack), {len(l)} live rows.")
    return t, l


def load_cic(files, cap=150000):
    """Loads and samples the CIC-IDS2017 day files into one frame (cap = total rows kept)."""
    per = max(5000, cap // max(1, len(files)))
    parts = []
    for f in files:
        d, _ = read_table(f, [])
        if len(d) > per:
            d = d.sample(n=per, random_state=SEED)
        print(f"[CIC-IDS2017] {os.path.basename(f)}: {len(d):,} rows kept")
        parts.append(d)
    return pd.concat(parts, ignore_index=True)

### Choose the dataset and prepare train / live data

In [ ]:
if DATASET == "nsl":
    NSL_KDD_COLUMNS, CATEGORICAL = list(NSL_ORIG_COLUMNS), list(NSL_ORIG_CAT)
    train_all, live_df = load_nsl(NSL_ORIG_COLUMNS, TRAIN_SAMPLE, SEED)
    train_df = train_all
else:
    if CIC_MODE == "upload":
        from google.colab import files as _f
        up = _f.upload()
        cic_files = list(up.keys())
    else:
        if not os.path.isdir(CIC_FOLDER):
            from google.colab import drive
            drive.mount("/content/drive")
        cic_files = sorted(glob.glob(os.path.join(CIC_FOLDER, "**", "*.csv"), recursive=True))
    assert cic_files, "No CSV files found. Use the TrafficLabelling version of CIC-IDS2017."
    raw = load_cic(cic_files)
    tr, feats, cats = build_frame(raw, "Label", ["benign"])
    if not cats:                                  # encoder needs at least one categorical column
        tr["_const"] = "all"; feats, cats = feats + ["_const"], ["_const"]
    shuf = tr.sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    k = int(len(shuf) * 0.7)
    train_df = shuf.iloc[:k].sample(n=min(TRAIN_SAMPLE, k), random_state=SEED).reset_index(drop=True)
    live_df = shuf.iloc[k:].reset_index(drop=True)
    NSL_KDD_COLUMNS, CATEGORICAL = feats + ["label"], list(cats)
    print(f"[Data] CIC-IDS2017: {len(train_df)} train rows ({int(train_df['binary_label'].sum())} attack), {len(live_df)} live rows.")
HAS_IPS = "_src" in live_df.columns and "_dst" in live_df.columns
print("Dataset has host IPs (network map available):", HAS_IPS)

## 2. Preprocessor  (one-hot encode + log scale + standardise)

In [ ]:
class Preprocessor:
    def __init__(self):
        self.encoder = OneHotEncoder(handle_unknown="ignore", sparse_output=False)
        self.scaler = StandardScaler()
        self.numeric_cols: List[str] = []
        self.feature_names: List[str] = []
        self.fitted = False

    def fit(self, df: pd.DataFrame):
        self.numeric_cols = [c for c in NSL_KDD_COLUMNS if c not in CATEGORICAL + ["label", "difficulty"]]
        self.encoder.fit(df[CATEGORICAL])
        num = np.log1p(df[self.numeric_cols].astype(float).clip(lower=0).values)  # tame heavy tails
        self.scaler.fit(num)
        self.feature_names = self.numeric_cols + list(self.encoder.get_feature_names_out(CATEGORICAL))
        self.fitted = True
        self.attack_mask = np.r_[np.ones(len(self.numeric_cols)), np.zeros(len(self.feature_names) - len(self.numeric_cols))]
        print(f"[Preprocessor] Fitted. {len(self.feature_names)} input features after encoding.")
        return self

    def transform(self, df: pd.DataFrame) -> np.ndarray:
        num = np.log1p(df[self.numeric_cols].astype(float).clip(lower=0).values)
        num = self.scaler.transform(num)
        cat = self.encoder.transform(df[CATEGORICAL])
        return np.hstack([num, cat]).astype(np.float64)

    # names kept from the UML diagram
    def normalize(self, df): return self.transform(df)
    def encodeFeatures(self, df): return self.transform(df)

## 3. IDSModel  (small neural network in NumPy, so we can compute REAL gradients for FGSM)

In [ ]:
class IDSModel:
    """MLP: input -> 64 -> 32 -> 1 (sigmoid). Binary: 0 = normal, 1 = attack."""

    def __init__(self, hidden=(64, 32), lr=0.003, epochs=25, batch=256, seed=SEED):
        self.hidden, self.lr, self.epochs, self.batch = hidden, lr, epochs, batch
        self.rng = np.random.default_rng(seed)
        self.weights: Optional[List[np.ndarray]] = None   # [W1,b1,W2,b2,W3,b3]
        self.isRobust: bool = False

    # ---- helpers ----
    def _init(self, d):
        sizes = [d, *self.hidden, 1]
        w = []
        for i in range(len(sizes) - 1):
            w.append(self.rng.normal(0, np.sqrt(2 / sizes[i]), (sizes[i], sizes[i + 1])))
            w.append(np.zeros(sizes[i + 1]))
        self.weights = w

    def _forward(self, X):
        W1, b1, W2, b2, W3, b3 = self.weights
        z1 = X @ W1 + b1; a1 = np.maximum(z1, 0)
        z2 = a1 @ W2 + b2; a2 = np.maximum(z2, 0)
        z3 = a2 @ W3 + b3
        p = 1 / (1 + np.exp(-np.clip(z3, -30, 30)))
        return z1, a1, z2, a2, p

    def _backward(self, X, y, cache):
        """Returns weight gradients and the gradient of the loss w.r.t. the INPUT X."""
        W1, b1, W2, b2, W3, b3 = self.weights
        z1, a1, z2, a2, p = cache
        n = len(X)
        dz3 = (p - y.reshape(-1, 1))                 # d(BCE)/dz3 (per sample)
        dW3 = a2.T @ dz3 / n; db3 = dz3.mean(0)
        da2 = dz3 @ W3.T; dz2 = da2 * (z2 > 0)
        dW2 = a1.T @ dz2 / n; db2 = dz2.mean(0)
        da1 = dz2 @ W2.T; dz1 = da1 * (z1 > 0)
        dW1 = X.T @ dz1 / n; db1 = dz1.mean(0)
        dX = dz1 @ W1.T                              # per-sample input gradient
        return [dW1, db1, dW2, db2, dW3, db3], dX

    # ---- public API ----
    def train(self, X: np.ndarray, y: np.ndarray, warm_start: bool = False,
              adv_eps: float = 0.0, mask: Optional[np.ndarray] = None):
        """adv_eps > 0 turns on adversarial training: every mini-batch is attacked with FGSM
        (against the CURRENT weights) and the model learns from clean + attacked copies."""
        tag = f", adversarial training eps={adv_eps}" if adv_eps > 0 else ""
        print(f"[IDSModel] Training on {len(X)} samples ({int(y.sum())} attack{tag})...")
        if self.weights is None or not warm_start:
            self._init(X.shape[1])
        m = [np.zeros_like(w) for w in self.weights]; v = [np.zeros_like(w) for w in self.weights]
        t = 0
        for ep in range(self.epochs):
            idx = self.rng.permutation(len(X))
            for s in range(0, len(X), self.batch):
                b = idx[s:s + self.batch]
                xb, yb = X[b], y[b]
                if adv_eps > 0:
                    g = self.input_gradient(xb, yb)
                    step = adv_eps * np.sign(g)
                    if mask is not None: step = step * mask
                    xb, yb = np.vstack([xb, xb + step]), np.concatenate([yb, yb])
                grads, _ = self._backward(xb, yb, self._forward(xb))
                t += 1
                for i, g in enumerate(grads):          # Adam
                    m[i] = 0.9 * m[i] + 0.1 * g
                    v[i] = 0.999 * v[i] + 0.001 * g * g
                    mh, vh = m[i] / (1 - 0.9 ** t), v[i] / (1 - 0.999 ** t)
                    self.weights[i] -= self.lr * mh / (np.sqrt(vh) + 1e-8)
        return self

    def predict_proba(self, X: np.ndarray) -> np.ndarray:
        return self._forward(X)[-1].ravel()           # P(attack)

    def predict(self, X: np.ndarray) -> np.ndarray:
        return (self.predict_proba(X) >= 0.5).astype(int)

    def confidence(self, X: np.ndarray) -> np.ndarray:
        p = self.predict_proba(X)
        return np.maximum(p, 1 - p)

    def input_gradient(self, X: np.ndarray, y: np.ndarray) -> np.ndarray:
        """dLoss/dInput for each sample (what FGSM needs)."""
        _, dX = self._backward(X, y, self._forward(X))
        return dX

## 4. FGSMGenerator  (real Fast Gradient Sign Method: x_adv = x + eps * sign(dLoss/dx))

In [ ]:
class FGSMGenerator:
    def __init__(self, epsilon: float = EPSILON, mask: Optional[np.ndarray] = None):
        self.epsilon = epsilon
        self.mask = mask          # 1 for features an attacker may change (numeric), 0 for one-hot flags

    def generateAdversarial(self, X: np.ndarray, y: np.ndarray, model: IDSModel) -> np.ndarray:
        print(f"[FGSMGenerator] Crafting {len(X)} adversarial samples with real gradients (eps={self.epsilon})...")
        step = self.epsilon * np.sign(model.input_gradient(X, y))
        if self.mask is not None: step = step * self.mask
        return X + step

## 5. AdversarialTrainer

In [ ]:
class AdversarialTrainer:
    def combineData(self, Xc, yc, Xa, ya):
        X, y = np.vstack([Xc, Xa]), np.concatenate([yc, ya])
        print(f"[AdversarialTrainer] Combined {len(Xc)} clean + {len(Xa)} adversarial = {len(X)}")
        return X, y

    def trainRobustModel(self, model: IDSModel, X, y, warm_start=False, adv_eps=0.0, mask=None) -> IDSModel:
        model.train(X, y, warm_start=warm_start, adv_eps=adv_eps, mask=mask)
        model.isRobust = True
        print("[AdversarialTrainer] Produced robust version of IDSModel.")
        return model

## 6. Evaluator

In [ ]:
class Evaluator:
    def testModel(self, model, X, y, name="model") -> Dict:
        pred = model.predict(X)
        r = {"name": name, "accuracy": round(accuracy_score(y, pred), 4),
             "precision": round(precision_score(y, pred, zero_division=0), 4),
             "recall": round(recall_score(y, pred, zero_division=0), 4),
             "f1": round(f1_score(y, pred, zero_division=0), 4), "isRobust": model.isRobust}
        print(f"[Evaluator] {r}")
        return r

    def compareResults(self, baseline: Dict, robust: Dict, key="accuracy") -> Dict:
        c = {f"baseline_{key}": baseline[key], f"robust_{key}": robust[key],
             "improvement": round(robust[key] - baseline[key], 4)}
        print(f"[Evaluator] Comparison: {c}")
        return c

## 7. Explanation: ExplanationReport + GenAIExplainer (real LLM if a key is set, otherwise a local explainer that uses the actual feature values)

In [ ]:
@dataclass
class ExplanationReport:
    attackType: str = ""
    explanationText: str = ""
    recommendedAction: str = ""
    timestamp: str = field(default_factory=lambda: time.strftime("%Y-%m-%d %H:%M:%S"))

    def toSummary(self) -> str:
        return (f"[{self.timestamp}] Attack='{self.attackType}' | "
                f"Explanation='{self.explanationText}' | Action='{self.recommendedAction}'")


class GenAIExplainer:
    def __init__(self, llmModel: str = "auto",
                 promptTemplate: str = ("You are a SOC analyst assistant. A network intrusion detector flagged a "
                                        "connection it is unsure about.\nPrediction: {label} (confidence {conf}).\n"
                                        "Most unusual features (value, z-score vs normal traffic): {features}.\n"
                                        "In 3 short sentences: say what kind of attack this resembles, why, and what "
                                        "the analyst should do. Do not invent facts beyond the numbers given.")):
        self.promptTemplate = promptTemplate
        self.gemini_key = os.environ.get("GEMINI_API_KEY", "")
        if llmModel == "auto":
            llmModel = ("gemini" if self.gemini_key else "local-rule-explainer")
        self.llmModel = llmModel
        self.normal_mean: Optional[np.ndarray] = None
        self.normal_std: Optional[np.ndarray] = None
        self.feature_names: List[str] = []

    def fit_baseline(self, X_normal: np.ndarray, feature_names: List[str]):
        self.normal_mean, self.normal_std = X_normal.mean(0), X_normal.std(0) + 1e-6
        self.feature_names = feature_names

    def _top_features(self, x: np.ndarray, k=4):
        z = (x - self.normal_mean) / self.normal_std
        order = np.argsort(-np.abs(z))[:k]
        return [(self.feature_names[i], float(x[i]), float(z[i])) for i in order]

    def _call_llm(self, prompt: str) -> Optional[str]:
        try:
            if self.llmModel == "gemini":
                url = ("https://generativelanguage.googleapis.com/v1beta/models/"
                       f"{LLM_MODEL_GEMINI}:generateContent?key={self.gemini_key}")
                body = json.dumps({"contents": [{"parts": [{"text": prompt}]}]}).encode()
                req = urllib.request.Request(url, body, {"content-type": "application/json"})
                return json.load(urllib.request.urlopen(req, timeout=30))["candidates"][0]["content"]["parts"][0]["text"].strip()
        except Exception as e:                      # network / key / model errors -> fall back
            print(f"[GenAIExplainer] LLM call failed ({type(e).__name__}); using local explainer.")
        return None

    def _local(self, feats, label, conf) -> str:
        parts = [f"{n}={v:.2f} ({'+' if z > 0 else ''}{z:.1f} sd)" for n, v, z in feats]
        names = " ".join(n for n, _, _ in feats)
        if any(k in names for k in ("serror", "count", "srv_count")):
            kind = "a flood / DoS style pattern (many connections or SYN errors)"
        elif any(k in names for k in ("rerror", "diff_srv", "dst_host_diff")):
            kind = "a probing / scanning pattern (many services or rejected connections)"
        elif any(k in names for k in ("failed_logins", "guest", "hot", "root", "compromised")):
            kind = "a remote access / privilege abuse pattern"
        else:
            kind = "an unfamiliar traffic pattern"
        return (f"Detector says '{label}' with only {conf:.0%} confidence. The most unusual features are "
                f"{', '.join(parts)}. This resembles {kind}.")

    def explainAttack(self, x: np.ndarray, prediction: Dict) -> str:
        feats = self._top_features(x)
        label = "attack" if prediction["prediction"] == 1 else "normal"
        text = None
        if self.llmModel in ("gemini",):
            fs = "; ".join(f"{n}={v:.2f} (z={z:+.1f})" for n, v, z in feats)
            text = self._call_llm(self.promptTemplate.format(label=label, conf=prediction["confidence"], features=fs))
        if text is None:
            text = self._local(feats, label, prediction["confidence"])
        print(f"[GenAIExplainer] ({self.llmModel}) Generated explanation.")
        return text

    def summarizePatch(self, cycle: int, recs: List[dict]) -> str:
        """GenAI summary of what was broken in this cycle and what the self-healer patched."""
        fixed = [r for r in recs if r["status"] == "PATCHED (was wrong)"]
        firm = [r for r in recs if r["status"] == "HARDENED (was right but unsure)"]
        bad = [r for r in recs if r["status"] == "STILL WRONG"]
        kinds = pd.Series([r["attack_type"] for r in recs]).value_counts().to_dict() if recs else {}
        facts = (f"Cycle {cycle}: {len(recs)} flagged. {len(fixed)} were misclassified and are now correct, "
                 f"{len(firm)} were correct but low-confidence and are now confident, {len(bad)} are still wrong. "
                 f"Attack families seen: {kinds}.")
        if self.llmModel in ("gemini",) and recs:
            t = self._call_llm("You are a SOC analyst assistant. Write a 3-sentence patch note for a security "
                               "team from these facts, no invented details: " + facts)
            if t: return t
        return facts

    def generateReport(self, x: np.ndarray, explanation: str) -> ExplanationReport:
        low = explanation.lower()
        kind = ("DoS / Flood" if "flood" in low or "dos" in low else
                "Probe / Scan" if "prob" in low or "scan" in low else
                "Remote access / Privilege abuse" if "privilege" in low or "login" in low else "Unknown / Zero-Day")
        return ExplanationReport(kind, explanation,
                                 "Isolate the source host, review logs, and approve the sample for retraining.")

## 8. IncidentKnowledgeBase + AttackMonitor

In [ ]:
class IncidentKnowledgeBase:
    def __init__(self):
        self.incidents: List[dict] = []
        self.unlabeledSamples: List[dict] = []

    def addIncident(self, sample: dict, report: ExplanationReport):
        self.incidents.append({"sample": sample, "report": report})
        self.unlabeledSamples.append(sample)

    def getUnlabeledSamples(self) -> List[dict]:
        return self.unlabeledSamples

    def clearProcessed(self):
        print(f"[IncidentKnowledgeBase] Cleared {len(self.unlabeledSamples)} processed samples.")
        self.unlabeledSamples = []


class AttackMonitor:
    def __init__(self, anomalyThreshold: float = CONF_THRESHOLD):
        self.liveTrafficBuffer: List[dict] = []
        self.anomalyThreshold = anomalyThreshold
        self.knownAttackSignatures: List[str] = []

    def monitorPredictions(self, predictions: List[dict]):
        self.liveTrafficBuffer.extend(predictions)

    def detectUnknownAttack(self, prediction: dict) -> bool:
        return prediction.get("confidence", 1.0) < self.anomalyThreshold

## 9. SelfHealingController  (detect -> explain -> log -> retrain -> deploy, repeated)

In [ ]:
class SelfHealingController:
    def __init__(self, cycleIntervalSec: int = 0):
        self.isActive = False
        self.cycleIntervalSec = cycleIntervalSec
        self.lastCycleTimestamp = ""
        self.history: List[dict] = []
        self.patch_log: List[dict] = []       # one row per flagged sample: what was broken, what fixed it
        self.patch_notes: List[str] = []      # GenAI patch note per cycle

    def logIncident(self, sample, report, kb):
        kb.addIncident(sample, report)

    def triggerRetraining(self, samples, trainer, model, Xtrain, ytrain, fgsm):
        """Retrain on: original training data + analyst-confirmed new samples, with adversarial training on top."""
        Xn = np.array([s["x"] for s in samples]); yn = np.array([s["label"] for s in samples])
        Xc, yc = np.vstack([Xtrain, Xn]), np.concatenate([ytrain, yn])
        print(f"[SelfHealingController] Retraining with {len(samples)} new incident samples...")
        return trainer.trainRobustModel(model, Xc, yc, adv_eps=fgsm.epsilon, mask=fgsm.mask), Xc, yc

    def deployModel(self, model):
        self.lastCycleTimestamp = time.strftime("%Y-%m-%d %H:%M:%S")
        print(f"[SelfHealingController] Auto-deployed updated model at {self.lastCycleTimestamp}.")

    def runFeedbackLoop(self, model, monitor, explainer, kb, trainer, fgsm, evaluator,
                        Xtrain, ytrain, X_live, y_live, cycle_no=1, max_explain=5):
        self.isActive = True
        print(f"\n[SelfHealingController] === Feedback loop #{cycle_no} ===")
        conf = model.confidence(X_live); pred = model.predict(X_live)
        preds = [{"prediction": int(p), "confidence": float(c)} for p, c in zip(pred, conf)]
        monitor.monitorPredictions(preds)

        acc_before = accuracy_score(y_live, pred)
        flagged = [i for i, p in enumerate(preds) if monitor.detectUnknownAttack(p)]
        print(f"[AttackMonitor] {len(flagged)} of {len(X_live)} live samples flagged as unknown (conf < {monitor.anomalyThreshold}).")

        pending = []
        if flagged:
            for n, i in enumerate(flagged):
                sample = {"x": X_live[i], "label": int(y_live[i])}    # label = analyst-confirmed (from dataset)
                if n < max_explain or explainer.llmModel not in ("gemini",):   # cap only real LLM calls (cost)
                    report = explainer.generateReport(X_live[i], explainer.explainAttack(X_live[i], preds[i]))
                    if n == 0: print("   example ->", report.toSummary())
                else:
                    report = ExplanationReport("Unknown", "Auto-logged (explanation skipped to save LLM calls).",
                                               "Queue for retraining.")
                self.logIncident(sample, report, kb)
                pending.append((i, report))
            print(f"[IncidentKnowledgeBase] Logged {len(flagged)} incidents. Total: {len(kb.incidents)}")
            model, Xnew, ynew = self.triggerRetraining(kb.getUnlabeledSamples(), trainer, model, Xtrain, ytrain, fgsm)
            self.deployModel(model)
            kb.clearProcessed()
            Xtrain, ytrain = Xnew, ynew                                # healed model keeps the new knowledge
        else:
            print("[SelfHealingController] Nothing flagged. No action needed.")

        # ---- what was broken, and was it patched? (compare the SAME samples before/after healing) ----
        if pending:
            idx = [i for i, _ in pending]
            p2, c2 = model.predict(X_live[idx]), model.confidence(X_live[idx])
            recs = []
            for k, (i, rep_) in enumerate(pending):
                truth, was_ok, now_ok = int(y_live[i]), preds[i]["prediction"] == int(y_live[i]), int(p2[k]) == int(y_live[i])
                status = ("PATCHED (was wrong)" if (not was_ok and now_ok) else
                          "HARDENED (was right but unsure)" if (was_ok and now_ok) else
                          "STILL WRONG" if not now_ok else "REGRESSED")
                recs.append({"cycle": cycle_no, "sample_id": i, "attack_type": rep_.attackType,
                             "truth": "attack" if truth else "normal",
                             "before": "attack" if preds[i]["prediction"] else "normal", "conf_before": round(preds[i]["confidence"], 2),
                             "after": "attack" if p2[k] else "normal", "conf_after": round(float(c2[k]), 2),
                             "status": status, "explanation": rep_.explanationText})
            self.patch_log.extend(recs)
            note = explainer.summarizePatch(cycle_no, recs)
            self.patch_notes.append(note)
            print(f"[GenAI patch note] {note}")

        acc_after = accuracy_score(y_live, model.predict(X_live))
        self.history.append({"cycle": cycle_no, "flagged": len(flagged),
                             "live_acc_before": round(acc_before, 4), "live_acc_after": round(acc_after, 4)})
        print(f"[SelfHealingController] Accuracy on this live batch: {acc_before:.4f} -> {acc_after:.4f}")
        self.isActive = False
        return model, Xtrain, ytrain

## 10. Run the whole pipeline
Baseline IDS → FGSM attack → adversarial training → Random Forest reference → self-healing loop on live traffic.

In [ ]:
def run_pipeline(train_df, live_df, epochs=EPOCHS, eps=EPSILON, thr=CONF_THRESHOLD, cycles=CYCLES, batch=BATCH_SIZE):
    n_tr = int(len(train_df) * 0.8)
    tr_df, te_df = train_df.iloc[:n_tr].reset_index(drop=True), train_df.iloc[n_tr:].reset_index(drop=True)
    print(f"[Data] Split into {len(tr_df)} train / {len(te_df)} test records.")

    # --- preprocess + baseline ---
    prep = Preprocessor().fit(tr_df)
    X_train, y_train = prep.transform(tr_df), tr_df["binary_label"].values
    X_test, y_test = prep.transform(te_df), te_df["binary_label"].values
    ev = Evaluator()
    baseline = IDSModel(epochs=epochs, seed=SEED).train(X_train, y_train)
    base_clean = ev.testModel(baseline, X_test, y_test, "Baseline (clean)")

    # --- FGSM attack on the baseline ---
    fgsm = FGSMGenerator(eps, mask=prep.attack_mask)
    X_test_adv = fgsm.generateAdversarial(X_test, y_test, baseline)
    base_adv = ev.testModel(baseline, X_test_adv, y_test, "Baseline (FGSM)")

    # --- adversarial training -> robust model ---
    trainer = AdversarialTrainer()
    X_train_adv = fgsm.generateAdversarial(X_train, y_train, baseline)
    trainer.combineData(X_train, y_train, X_train_adv, y_train)
    robust = trainer.trainRobustModel(IDSModel(epochs=epochs, seed=SEED + 1), X_train, y_train,
                                      adv_eps=eps, mask=prep.attack_mask)
    rob_clean = ev.testModel(robust, X_test, y_test, "Robust (clean)")
    X_adv_rob = FGSMGenerator(eps, prep.attack_mask).generateAdversarial(X_test, y_test, robust)
    rob_adv = ev.testModel(robust, X_adv_rob, y_test, "Robust (FGSM own)")
    rob_tr = ev.testModel(robust, X_test_adv, y_test, "Robust (FGSM transfer)")
    ev.compareResults(base_adv, rob_adv)

    # --- reference Random Forest ---
    rf = RandomForestClassifier(n_estimators=100, max_depth=12, random_state=SEED, n_jobs=-1).fit(X_train, y_train)
    p = rf.predict(X_test)
    rf_row = {"name": "RandomForest (clean)", "accuracy": round(accuracy_score(y_test, p), 4),
              "precision": round(precision_score(y_test, p, zero_division=0), 4),
              "recall": round(recall_score(y_test, p, zero_division=0), 4), "f1": round(f1_score(y_test, p, zero_division=0), 4)}
    evals = [base_clean, base_adv, rob_clean, rob_adv, rob_tr, rf_row]

    # --- self-healing loop on live traffic ---
    explainer = GenAIExplainer()
    explainer.fit_baseline(X_train[y_train == 0], prep.feature_names)
    monitor, kb, ctrl = AttackMonitor(thr), IncidentKnowledgeBase(), SelfHealingController()
    X_live, y_live = prep.transform(live_df), live_df["binary_label"].values
    bs = max(50, min(batch, len(X_live) // (cycles + 1)))
    while cycles * bs > len(X_live) and cycles > 1:
        cycles -= 1
    print(f"[Live] {len(X_live)} rows -> {cycles} cycles x {bs} rows. Explainer: {explainer.llmModel}")
    live_model, Xtr, ytr = copy.deepcopy(robust), X_train, y_train
    for c in range(cycles):
        s, e = c * bs, (c + 1) * bs
        live_model, Xtr, ytr = ctrl.runFeedbackLoop(live_model, monitor, explainer, kb, trainer, fgsm, ev,
                                                    Xtr, ytr, X_live[s:e], y_live[s:e], cycle_no=c + 1)
    held = cycles * bs
    held_b = held_a = None
    if held < len(X_live) - 20:
        held_b = accuracy_score(y_live[held:], robust.predict(X_live[held:]))
        held_a = accuracy_score(y_live[held:], live_model.predict(X_live[held:]))
        print(f"[Result] Held-out live accuracy: {held_b:.4f} -> {held_a:.4f}")

    ipmap = None
    if "_src" in live_df.columns and "_dst" in live_df.columns:      # link IDS alerts to hosts
        ipmap = live_df[["_src", "_dst", "label", "binary_label"]].copy()
        ipmap["flag_before"] = np.asarray(robust.predict(X_live)).astype(int)
        ipmap["flag_after"] = np.asarray(live_model.predict(X_live)).astype(int)
    return dict(evals=evals, history=pd.DataFrame(ctrl.history), patch=pd.DataFrame(ctrl.patch_log), notes=ctrl.patch_notes,
                held=(held_b, held_a), incidents=len(kb.incidents), explainer=explainer.llmModel, ipmap=ipmap,
                model=live_model, prep=prep)

results = run_pipeline(train_df, live_df)

### Results: accuracy, FGSM robustness, self-healing history

In [ ]:
ev = {e["name"]: e for e in results["evals"]}
display(pd.DataFrame(results["evals"])[["name", "accuracy", "precision", "recall", "f1"]])

fig, ax = plt.subplots(figsize=(7, 3.6))
names = ["Baseline (clean)", "Baseline (FGSM)", "Robust (clean)", "Robust (FGSM own)"]
vals = [ev[n]["accuracy"] for n in names]
bars = ax.bar([n.replace(" (", "\n(") for n in names], vals, color=["#4c78a8", "#e45756", "#4c78a8", "#54a24b"])
ax.bar_label(bars, fmt="%.3f"); ax.set_ylim(0, 1.08); ax.set_ylabel("accuracy")
ax.set_title("IDS accuracy under FGSM attack (adversarial robustness)")
plt.show()

print("\nSelf-healing loop history:")
display(results["history"])
hb, ha = results["held"]
if hb is not None:
    print(f"Held-out live traffic: {hb:.2%} before healing -> {ha:.2%} after healing")
print("Incidents logged:", results["incidents"], "| Explainer used:", results["explainer"])
print("\nGenAI patch notes:")
for n in results["notes"]:
    print(" -", n)
if len(results["patch"]):
    print("\nSample of the healing report (what was broken, what was patched):")
    display(results["patch"].drop(columns=["explanation"]).head(10))

## 11. Network healing module
A network is modelled as nodes (routers, firewalls, switches, hosts) and links, some of them **standby (dark) links**.
`NetworkHealer` repairs it in 7 steps: **Detect → Contain → Fail over → Restore → Repair links → Verify → Harden**.
`build_ids_network` places the IDS alerts on the map: attacker IPs, victim hosts, and which attacks damaged which host.

In [ ]:
USABLE = ("healthy", "degraded", "restored")
STATE_COLOR = {"healthy": "#2ca02c", "degraded": "#f0ad4e", "compromised": "#d9534f",
               "failed": "#444444", "quarantined": "#8e5ea2", "restored": "#2e86de"}
DAMAGE_SCENARIOS = ["Node failures", "Link cuts", "Malware outbreak", "DDoS attack", "Ransomware", "Combined disaster"]
ROLE_ORDER = {"router": 0, "firewall": 1, "switch": 2, "server": 3, "host": 4, "workstation": 4}


class CyberNetwork:
    def __init__(self, core="ISP"):
        self.core = core
        self.nodes, self.links = {}, {}

    @staticmethod
    def key(a, b):
        return tuple(sorted((a, b)))

    def add_node(self, name, role, x=0.0, y=0.0, backup=True):
        self.nodes[name] = dict(role=role, state="healthy", backup=backup, x=x, y=y, data_lost=False)

    def add_link(self, a, b, standby=False):
        self.links[self.key(a, b)] = dict(up=True, active=not standby, failover=False)

    def copy(self):
        return copy.deepcopy(self)

    # ---- graph helpers ----
    def _bfs(self, usable_node, usable_link):
        if self.core not in self.nodes or not usable_node(self.nodes[self.core]):
            return set()
        adj = {n: [] for n in self.nodes}
        for (u, v), l in self.links.items():
            if usable_link(l) and usable_node(self.nodes[u]) and usable_node(self.nodes[v]):
                adj[u].append(v)
                adj[v].append(u)
        seen, dq = {self.core}, _deque([self.core])
        while dq:
            c = dq.popleft()
            for m in adj[c]:
                if m not in seen:
                    seen.add(m)
                    dq.append(m)
        return seen

    def reachable(self):
        """Nodes the core can reach through trusted (healthy/degraded/restored) nodes on live links."""
        return self._bfs(lambda d: d["state"] in USABLE, lambda l: l["up"] and l["active"])

    def mgmt_reach(self):
        """Management plane: reaches everything not dead, even compromised hosts."""
        return self._bfs(lambda d: d["state"] != "failed", lambda l: l["up"] and l["active"])

    def health(self):
        reach = self.reachable()
        real = [n for n, d in self.nodes.items() if n != self.core and d["role"] != "attacker"]
        good = [n for n in real if self.nodes[n]["state"] in ("healthy", "restored") and n in reach]
        return len(good) / max(1, len(real))

    def state_counts(self):
        c = {}
        for n, d in self.nodes.items():
            c[d["state"]] = c.get(d["state"], 0) + 1
        return c


def default_network():
    net = CyberNetwork("ISP")
    tiers = [[("ISP", "cloud")], [("R1", "router"), ("R2", "router")], [("FW1", "firewall"), ("FW2", "firewall")],
             [("SW1", "switch"), ("SW2", "switch"), ("SW3", "switch")],
             [("WEB", "server"), ("DB", "server"), ("MAIL", "server"), ("DNS", "server")],
             [(f"WS{i}", "workstation") for i in range(1, 9)]]
    for t, row in enumerate(tiers):
        k = len(row)
        for i, (name, role) in enumerate(row):
            net.add_node(name, role, 5.0 if k == 1 else 0.5 + 9.0 * i / (k - 1), -1.3 * t,
                         backup=name not in ("WS8", "DNS"))
    for a, b in [("ISP", "R1"), ("ISP", "R2"), ("R1", "FW1"), ("R2", "FW2"), ("FW1", "SW1"), ("FW1", "SW2"),
                 ("FW2", "SW2"), ("FW2", "SW3"), ("SW1", "WEB"), ("SW1", "DB"), ("SW2", "MAIL"), ("SW2", "DNS"),
                 ("SW2", "WS1"), ("SW2", "WS2"), ("SW3", "WS3"), ("SW3", "WS4"), ("SW3", "WS5"),
                 ("SW1", "WS6"), ("SW1", "WS7"), ("SW1", "WS8")]:
        net.add_link(a, b)
    for a, b in [("SW1", "SW3"), ("SW2", "DB"), ("SW1", "MAIL"), ("SW3", "DNS"), ("FW1", "FW2"), ("R1", "R2")]:
        net.add_link(a, b, standby=True)       # dark links the healer can light up
    return net


def network_from_edges(df):
    """Edge list CSV: first two columns = node A, node B. Optional column 'standby' (1/true = dark spare link)."""
    cols = list(df.columns)
    a, b = df[cols[0]].astype(str).str.strip(), df[cols[1]].astype(str).str.strip()
    sb = df["standby"].astype(str).str.lower().isin(["1", "true", "yes"]) if "standby" in df.columns else pd.Series(False, index=df.index)
    deg = pd.concat([a, b]).value_counts()
    names = list(deg.index)
    net = CyberNetwork(core=names[0])
    for i, n in enumerate(names):
        ang = 2 * np.pi * i / max(1, len(names))
        d = int(deg[n])
        role = "cloud" if n == net.core else "router" if d >= 4 else "switch" if d == 3 else "workstation"
        net.add_node(n, role, 5 + 4.2 * np.cos(ang), 4.2 * np.sin(ang))
    for x, y, s in zip(a, b, sb):
        if x != y:
            net.add_link(x, y, standby=bool(s))
    return net


def inflict_damage(net, scenario, size, rng):
    """Damages the network in place and returns a list of what happened."""
    log = []
    cand = [n for n in net.nodes if n != net.core and net.nodes[n]["state"] == "healthy"]
    live_links = [k for k, l in net.links.items() if l["up"] and l["active"]]

    def node_failures():
        for n in rng.sample(cand, min(size, len(cand))):
            net.nodes[n]["state"] = "failed"
            log.append(f"{n} ({net.nodes[n]['role']}) crashed")

    def link_cuts():
        for k in rng.sample(live_links, min(size, len(live_links))):
            net.links[k]["up"] = False
            log.append(f"link {k[0]}–{k[1]} cut")

    def malware():
        zero = rng.choice([n for n in cand if net.nodes[n]["role"] in ("workstation", "server", "host")] or cand)
        infected, frontier = {zero}, {zero}
        for _ in range(max(1, size - 1)):
            nxt = set()
            for (u, v), l in net.links.items():
                for s, t in ((u, v), (v, u)):
                    if s in frontier and t not in infected and t != net.core and rng.random() < 0.6:
                        nxt.add(t)
            infected |= nxt
            frontier = nxt
        for n in infected:
            net.nodes[n]["state"] = "compromised"
        log.append(f"malware started at {zero} and spread to {len(infected)} node(s): {sorted(infected)}")

    def ddos():
        tgt = [n for n in cand if net.nodes[n]["role"] in ("server", "router", "firewall")]
        for n in rng.sample(tgt, min(size, len(tgt))):
            net.nodes[n]["state"] = "degraded"
            log.append(f"{n} saturated by DDoS traffic")

    def ransom():
        for n in rng.sample(cand, min(size, len(cand))):
            net.nodes[n]["state"] = "compromised"
            net.nodes[n]["data_lost"] = True
            log.append(f"{n} encrypted by ransomware")

    {"Node failures": node_failures, "Link cuts": link_cuts, "Malware outbreak": malware,
     "DDoS attack": ddos, "Ransomware": ransom}.get(scenario, lambda: None)()
    if scenario == "Combined disaster":
        for fn in (link_cuts, node_failures, malware):
            cand[:] = [n for n in net.nodes if n != net.core and net.nodes[n]["state"] == "healthy"]
            live_links[:] = [k for k, l in net.links.items() if l["up"] and l["active"]]
            fn()
    return log


class NetworkHealer:
    """Autonomous network repair: detect -> contain -> fail over -> restore -> repair links -> verify -> harden."""
    BASE = dict(cpu=(30, 6), traffic=(120, 25), err=(0.5, 0.3))

    def __init__(self, seed=42):
        self.rng = random.Random(seed)
        self.actions = []

    def _log(self, phase, target, action, result, secs=0):
        self.actions.append(dict(phase=phase, target=target, action=action, result=result, seconds=secs))

    def _telemetry(self, state):
        r = self.rng.gauss
        if state == "degraded":
            return r(92, 4), r(900, 80), r(8, 1.5)
        if state in ("compromised", "quarantined"):
            return r(78, 8), r(520, 60), r(0.6, 0.3)
        if state == "failed":
            return 0.0, 0.0, 0.0
        return r(30, 6), r(120, 25), r(0.5, 0.3)

    # 1. DETECT -------------------------------------------------------------
    def detect(self, net, detected=None):
        """detected=None -> telemetry sees everything (manual sim). A set -> only hosts the IDS flagged are acted on."""
        mg, issues = net.mgmt_reach(), {}
        for n, d in net.nodes.items():
            if n == net.core:
                continue
            kinds = []
            cpu, trf, err = self._telemetry(d["state"])
            z = max((cpu - self.BASE["cpu"][0]) / self.BASE["cpu"][1],
                    (trf - self.BASE["traffic"][0]) / self.BASE["traffic"][1],
                    (err - self.BASE["err"][0]) / self.BASE["err"][1])
            if d["state"] == "failed":
                kinds.append("node failure")
            else:
                if n not in mg:
                    kinds.append("isolated (link fault)")
                if z > 4:
                    kinds.append("DDoS saturation" if err > 3 else "compromise")
            if kinds and detected is not None and n not in detected:
                self._log("1 Detect", n, "IDS alert check", "no IDS alert: fault NOT detected, left untouched", 1)
                continue
            if kinds:
                issues[n] = dict(kinds=kinds, z=z)
                self._log("1 Detect", n, "heartbeat + telemetry anomaly check",
                          ", ".join(kinds) + (f" (anomaly z={z:.1f})" if z > 4 else ""), 2)
        for k, l in net.links.items():
            if not l["up"]:
                self._log("1 Detect", f"{k[0]}–{k[1]}", "link-state probe", "link down", 1)
        if not issues and all(l["up"] for l in net.links.values()):
            self._log("1 Detect", "network", "full sweep", "no faults found", 2)
        return issues

    # 2. CONTAIN ------------------------------------------------------------
    def contain(self, net, issues):
        for n, i in issues.items():
            if "compromise" in i["kinds"]:
                net.nodes[n]["state"] = "quarantined"
                if net.nodes[n]["role"] == "attacker":
                    self._log("2 Contain", n, "block source IP at perimeter firewall", "attacker blocked", 3)
                else:
                    self._log("2 Contain", n, "quarantine: block all ports, revoke credentials", "spread stopped", 3)

    # 3. FAIL OVER ----------------------------------------------------------
    def reroute(self, net, phase="3 Fail over"):
        changed = True
        while changed:
            changed = False
            reach = net.reachable()
            for (u, v), l in net.links.items():
                if l["up"] and not l["active"] and net.nodes[u]["state"] in USABLE and net.nodes[v]["state"] in USABLE \
                        and ((u in reach) != (v in reach)):
                    l["active"], l["failover"] = True, True
                    lost = v if u in reach else u
                    self._log(phase, f"{u}–{v}", f"activate standby link to re-attach {lost}", "traffic rerouted", 5)
                    changed = True
                    break

    # 4. RESTORE ------------------------------------------------------------
    def restore(self, net, issues):
        order = sorted(issues, key=lambda n: ROLE_ORDER.get(net.nodes[n]["role"], 5))
        for n in order:
            d = net.nodes[n]
            if d["role"] == "attacker":
                continue                              # attackers stay blocked
            if d["state"] == "failed":
                d["state"] = "restored"
                self._log("4 Restore", n, "hard reset / fail over to hot spare", "back online", 30)
            elif d["state"] in ("quarantined", "compromised"):
                if d["backup"]:
                    self._log("4 Restore", n, "wipe, restore last clean backup, apply patches", "clean and rejoined", 90)
                else:
                    self._log("4 Restore", n, "no backup: rebuild from golden image, apply patches", "rebuilt and rejoined", 240)
                d["state"], d["data_lost"] = "restored", False
            elif d["state"] == "degraded":
                d["state"] = "healthy"
                self._log("4 Restore", n, "rate-limit, traffic scrubbing, SYN cookies", "load normal", 20)

    # 5. REPAIR LINKS -------------------------------------------------------
    def repair_links(self, net):
        for k, l in net.links.items():
            if not l["up"]:
                l["up"] = True
                self._log("5 Repair", f"{k[0]}–{k[1]}", "reset interface / switch to spare port", "link up", 60)

    # 7. HARDEN -------------------------------------------------------------
    def weak_points(self, net):
        base = len(net.reachable())
        out = []
        for n in net.nodes:
            if n == net.core or net.nodes[n]["role"] == "attacker":
                continue
            t = net.copy()
            t.nodes[n]["state"] = "failed"
            lost = [m for m in net.reachable() if m != n and m not in t.reachable()]
            if lost:
                out.append((len(lost), f"Node {n}: single point of failure, {len(lost)} node(s) lose service {sorted(lost)[:5]}. Add a redundant uplink."))
        for k, l in net.links.items():
            if l["up"] and l["active"]:
                t = net.copy()
                t.links[k]["up"] = False
                lost = [m for m in net.reachable() if m not in t.reachable()]
                if len(lost) >= 2:
                    out.append((len(lost), f"Link {k[0]}–{k[1]}: single point of failure, {len(lost)} node(s) cut off. Add a parallel link."))
        return [s for _, s in sorted(out, reverse=True)][:6]

    # full cycle -------------------------------------------------------------
    def heal(self, net, detected=None):
        self.actions = []
        h0 = net.health()
        issues = self.detect(net, detected)
        self.contain(net, issues)
        self.reroute(net)
        self.restore(net, issues)
        self.repair_links(net)
        self.reroute(net, "5 Repair")
        h1 = net.health()
        reach = net.reachable()
        resid = [n for n, d in net.nodes.items() if n != net.core and not (d["role"] == "attacker" and d["state"] == "quarantined")
                 and (d["state"] not in ("healthy", "restored") or n not in reach)]
        self._log("6 Verify", "network", "end-to-end reachability + health check",
                  f"health {h0:.0%} -> {h1:.0%}" + (f", residual: {resid}" if resid else ", all nodes healthy"), 5)
        weak = self.weak_points(net)
        for w in weak[:3]:
            self._log("7 Harden", "topology", "redundancy analysis", w, 0)
        return dict(before=h0, after=h1, actions=list(self.actions), issues=issues, residual=resid,
                    weak=weak, seconds=sum(a["seconds"] for a in self.actions))


def attack_effect(label):
    """What an attack does to its victim host. None = recon only (no damage)."""
    l = str(label).lower()
    if any(k in l for k in ("portscan", "port scan", "scan", "probe", "recon")):
        return None
    if any(k in l for k in ("dos", "ddos", "slowloris", "slowhttp", "hulk", "goldeneye", "flood")):
        return "degraded"
    return "compromised"


def build_ids_network(ipdf, n_hosts=12, n_att=4):
    """Builds the network map from real IDS results. ipdf columns: _src,_dst,label,binary_label,flag_before,flag_after.
    Returns None when there is no attack traffic to show."""
    att = ipdf[ipdf["binary_label"] == 1]
    if att.empty:
        return None
    attackers = list(att["_src"].value_counts().index[:n_att])
    hosts = list(att[~att["_dst"].isin(attackers)]["_dst"].value_counts().index[:n_hosts])
    for h in ipdf[ipdf["binary_label"] == 0]["_dst"].value_counts().index:      # pad with normal hosts
        if len(hosts) >= n_hosts:
            break
        if h not in hosts and h not in attackers:
            hosts.append(h)
    net = CyberNetwork("INTERNET")
    net.add_node("INTERNET", "cloud", 5.0, -1.3)
    for i, a in enumerate(attackers):
        net.add_node(a, "attacker", 5.0 if len(attackers) == 1 else 0.5 + 9.0 * i / (len(attackers) - 1), 0.0)
        net.add_link(a, "INTERNET")
    for name, role, x, y in [("FW1", "firewall", 3.0, -2.6), ("FW2", "firewall", 7.0, -2.6),
                             ("SW1", "switch", 3.0, -3.9), ("SW2", "switch", 7.0, -3.9)]:
        net.add_node(name, role, x, y)
    net.add_link("INTERNET", "FW1")
    net.add_link("INTERNET", "FW2")
    for sw in ("SW1", "SW2"):
        net.add_link("FW1", sw)
        net.add_link("FW2", sw, standby=True)
    net.add_link("SW1", "SW2", standby=True)
    for i, h in enumerate(hosts):
        net.add_node(h, "host", 0.5 if len(hosts) == 1 else 0.5 + 9.0 * i / (len(hosts) - 1), -5.4 - 0.5 * (i % 2))
        net.add_link(h, "SW1" if i % 2 == 0 else "SW2")
        net.add_link(h, "SW2" if i % 2 == 0 else "SW1", standby=True)
    dmg = net.copy()
    for a in attackers:
        dmg.nodes[a]["state"] = "compromised"
    for dst, lab in att[["_dst", "label"]].drop_duplicates().itertuples(index=False):
        eff = attack_effect(lab)
        if eff and dst in dmg.nodes and dmg.nodes[dst]["role"] != "attacker":
            cur = dmg.nodes[dst]["state"]
            dmg.nodes[dst]["state"] = "compromised" if (eff == "compromised" or cur == "compromised") else "degraded"
    hit = att[att["flag_after"] == 1]
    hit_b = att[att["flag_before"] == 1]
    det, det_b = set(hit["_dst"]) | set(hit["_src"]), set(hit_b["_dst"]) | set(hit_b["_src"])
    affected = {n for n, d in dmg.nodes.items() if d["state"] != "healthy"}
    return dict(base=net, damaged=dmg, detected=det, affected=affected,
                caught_before=len(affected & det_b), caught_after=len(affected & det),
                flows=len(ipdf), attack_flows=len(att), labels=att["label"].value_counts().head(5).to_dict())


def load_cic(files, cap=150000):
    """Loads and samples the CIC-IDS2017 day files into one frame (cap = total rows kept)."""
    per = max(5000, cap // max(1, len(files)))
    parts = []
    for f in files:
        d, _ = read_table(f, [])
        if len(d) > per:
            d = d.sample(n=per, random_state=SEED)
        print(f"[CIC-IDS2017] {os.path.basename(f)}: {len(d):,} rows kept")
        parts.append(d)
    return pd.concat(parts, ignore_index=True)


def network_report(info, rep, base, dmg, healed, explainer_name, loop_notes=None, mode="ids"):
    """Plain-language report of what happened and what the self-healer did (shown in the GenAI explainer box)."""
    victims = [n for n, d in dmg.nodes.items() if d["state"] != "healthy" and d["role"] != "attacker"]
    attackers = [n for n, d in dmg.nodes.items() if d["role"] == "attacker" and d["state"] != "healthy"]
    acts = rep["actions"]
    cnt = lambda key: sum(1 for a in acts if key in a["result"] or key in a["action"])
    kinds = {}
    for n in victims:
        kinds.setdefault(dmg.nodes[n]["state"], []).append(n)
    L = [f"INCIDENT & SELF-HEALING REPORT   (explainer: {explainer_name})", ""]
    L.append("1. WHAT HAPPENED")
    if mode == "ids" and info:
        lab = ", ".join(f"{k} ({v:,})" for k, v in info["labels"].items())
        L.append(f"   {info['flows']:,} live flows were analysed; {info['attack_flows']:,} were attacks: {lab}.")
    if attackers:
        L.append(f"   Attacking sources: {', '.join(attackers)}.")
    for st, ns in kinds.items():
        verb = {"compromised": "were compromised", "degraded": "were degraded (overloaded)", "failed": "went down"}.get(st, st)
        L.append(f"   {len(ns)} host(s) {verb}: {', '.join(ns[:8])}{'...' if len(ns) > 8 else ''}.")
    cut = [k for k, l in dmg.links.items() if not l["up"]]
    if cut:
        L.append(f"   {len(cut)} link(s) were cut: {', '.join(a + '–' + b for a, b in cut[:5])}.")
    L.append(f"   Network health fell from {base.health():.0%} to {dmg.health():.0%}.")
    L += ["", "2. HOW THE IDS RESPONDED"]
    if mode == "ids" and info:
        n = len(info["affected"])
        L.append(f"   Before self-healing the IDS flagged {info['caught_before']}/{n} affected hosts; after the self-healing "
                 f"loop retrained the model it flagged {info['caught_after']}/{n}.")
        if loop_notes:
            L.append("   Model patch note: " + str(loop_notes[0])[:300])
    else:
        L.append("   Faults were found by heartbeat checks and telemetry anomaly scoring.")
    L += ["", "3. WHAT THE SELF-HEALER DID"]
    steps = [("attacker blocked", "blocked {} attacker IP(s) at the perimeter firewall"),
             ("spread stopped", "quarantined {} compromised host(s) so the infection could not spread"),
             ("back online", "reset {} failed device(s) / switched to hot spare"),
             ("clean and rejoined", "wiped and restored {} host(s) from clean backup, then patched them"),
             ("rebuilt and rejoined", "rebuilt {} host(s) from the golden image (no backup existed)"),
             ("load normal", "applied rate-limiting and traffic scrubbing on {} overloaded host(s)"),
             ("traffic rerouted", "activated {} standby link(s) to reroute traffic"),
             ("link up", "repaired {} link(s)")]
    for key, txt in steps:
        c = sum(1 for a in acts if a["result"] == key)
        if c:
            L.append("   - " + txt.format(c))
    L.append(f"   Estimated repair time: {rep['seconds']} s.")
    L += ["", "4. RESULT", f"   Network health {rep['before']:.0%} -> {rep['after']:.0%}."]
    if rep["residual"]:
        L.append(f"   Still damaged: {', '.join(rep['residual'][:8])}. These were not flagged by the IDS, so no action was taken on them.")
    else:
        L.append("   All hosts are healthy again" + ("; attackers remain blocked." if attackers else "."))
    L += ["", "5. RECOMMENDATIONS"]
    rec = []
    if attackers:
        rec.append("Keep the blocked attacker IPs on the firewall blocklist and review how they got in.")
    if "compromised" in kinds:
        rec.append("Rotate credentials and verify patch level on the restored hosts.")
    if "degraded" in kinds:
        rec.append("Enable upstream DDoS protection and rate limits on the affected services.")
    if rep["residual"]:
        rec.append("Add the missed hosts' flows to the training data and run another self-healing cycle.")
    rec += rep["weak"][:2]
    L += [f"   - {r}" for r in (rec or ["No further action needed."])]
    return "\n".join(L)


def draw_network(ax, net, title):
    ax.clear()
    ax.set_title(title, fontsize=10, color=PRIMARY)
    ax.axis("off")
    if net is None:
        ax.text(0.5, 0.5, "–", ha="center", va="center", transform=ax.transAxes, color=MUTED)
        return
    for (u, v), l in net.links.items():
        x, y = [net.nodes[u]["x"], net.nodes[v]["x"]], [net.nodes[u]["y"], net.nodes[v]["y"]]
        if not l["up"]:
            kw = dict(color="#d9534f", ls="--", lw=1.8)
        elif not l["active"]:
            kw = dict(color="#cfd6de", ls=":", lw=1.0)
        elif l["failover"]:
            kw = dict(color="#2e86de", lw=2.2)
        else:
            kw = dict(color="#9aa5b1", lw=1.2)
        ax.plot(x, y, zorder=1, **kw)
    for n, d in net.nodes.items():
        ax.scatter(d["x"], d["y"], s=150, color=STATE_COLOR[d["state"]], edgecolor="white", zorder=2)
        ax.text(d["x"], d["y"] - 0.33, n, ha="center", va="top", fontsize=6.5)
    ax.margins(0.08)

In [ ]:
def show_network(base, dmg, healed, title=""):
    fig, axes = plt.subplots(1, 3, figsize=(15, 4.4))
    draw_network(axes[0], base, "1 · Intact network")
    draw_network(axes[1], dmg, "2 · After damage / attack")
    draw_network(axes[2], healed, "3 · After self-healing")
    hs = [Line2D([], [], marker="o", ls="", color=c, label=k) for k, c in STATE_COLOR.items()]
    hs += [Line2D([], [], color="#d9534f", ls="--", label="cut link"), Line2D([], [], color="#2e86de", lw=2, label="failover link"),
           Line2D([], [], color="#cfd6de", ls=":", label="standby (dark)")]
    fig.legend(handles=hs, loc="lower center", ncol=9, fontsize=8, frameon=False)
    fig.suptitle(title, color=PRIMARY)
    fig.subplots_adjust(bottom=0.1, top=0.88)
    plt.show()

## 12. Heal the network from the IDS results
If the dataset has IP addresses (CIC-IDS2017) the map is built from the real alerts and healing runs automatically.
For datasets without IPs (NSL-KDD) there is nothing to map, so this section says so.

In [ ]:
net_info = build_ids_network(results["ipmap"]) if results["ipmap"] is not None else None
if net_info is None:
    print("Nothing to show: this dataset has no source/destination IP columns, so attacks cannot be placed on a network.")
    print("Use DATASET = 'cic' (CIC-IDS2017 TrafficLabelling) for the linked demo, or run the manual simulation below.")
else:
    base_net, dmg_net = net_info["base"], net_info["damaged"]
    healed_net = dmg_net.copy()
    rep = NetworkHealer().heal(healed_net, net_info["detected"])
    print(f"Network health: intact {base_net.health():.0%} -> after attacks {dmg_net.health():.0%} -> after healing {rep['after']:.0%}")
    show_network(base_net, dmg_net, healed_net, "Network healing driven by IDS alerts")
    display(pd.DataFrame(rep["actions"]))

### GenAI explainer: incident and self-healing report

In [ ]:
if net_info is not None:
    explainer = GenAIExplainer()
    report_text = network_report(net_info, rep, base_net, dmg_net, healed_net, explainer.llmModel,
                                 results["notes"], "ids")
    if explainer.llmModel == "gemini":
        ai = explainer._call_llm("You are a SOC analyst assistant. Write a concise incident report (max 170 words, plain prose) "
                                 "for a security team from these facts only, inventing nothing:\n" + report_text)
        if ai:
            report_text = f"AI SUMMARY (gemini)\n{ai}\n\n" + "-" * 40 + "\n" + report_text
    print(report_text)
else:
    print("No report: nothing to show for this dataset (no IP addresses).")

## 13. Manual simulation (works with any dataset)
Damage a built-in 21-node enterprise network and watch the healer repair it. Change the scenario to try each one:
`Node failures`, `Link cuts`, `Malware outbreak`, `DDoS attack`, `Ransomware`, `Combined disaster`.

In [ ]:
SCENARIO, SIZE = "Combined disaster", 3

demo_base = default_network()
demo_dmg = demo_base.copy()
what = inflict_damage(demo_dmg, SCENARIO, SIZE, random.Random())
print("Damage:", "; ".join(what))
demo_healed = demo_dmg.copy()
demo_rep = NetworkHealer().heal(demo_healed)
print(f"Health: intact {demo_base.health():.0%} -> damaged {demo_dmg.health():.0%} -> healed {demo_rep['after']:.0%}  "
      f"(estimated repair time {demo_rep['seconds']} s)")
show_network(demo_base, demo_dmg, demo_healed, f"Manual simulation: {SCENARIO}")
display(pd.DataFrame(demo_rep["actions"]))
print()
print(network_report(None, demo_rep, demo_base, demo_dmg, demo_healed, GenAIExplainer().llmModel, None, "manual"))

## Summary
- The neural IDS detects attacks; FGSM shows how easily a normal model is fooled, and adversarial training makes it robust.
- The self-healing loop finds low-confidence (possibly new) attacks, explains them, retrains and redeploys automatically.
- The network healer isolates compromised hosts, reroutes traffic over standby links, restores nodes from backup and verifies health.
- The GenAI explainer turns every step into a plain-language report.

**Limitations:** the network is simulated; no real devices are touched. Label-leaking columns are removed so accuracy reflects real detection.